# Spotify Track Data Retrieval
This notebook collects Spotify track data using the Spotify Web API via the Spotipy library. 
The dataset includes numeric, categorical, and text features for use in predictive analytics.

In [1]:
# Install required libraries
%pip install spotipy python-dotenv pandas


   ---------------------------------------- 0/3 [redis]
   ---------------------------------------- 0/3 [redis]
   ---------------------------------------- 0/3 [redis]
   ---------------------------------------- 0/3 [redis]
   ---------------------------------------- 0/3 [redis]
   ---------------------------------------- 0/3 [redis]
   ---------------------------------------- 0/3 [redis]
   ---------------------------------------- 0/3 [redis]
   ---------------------------------------- 0/3 [redis]
   ---------------------------------------- 0/3 [redis]
   ---------------------------------------- 0/3 [redis]
   ------------- -------------------------- 1/3 [python-dotenv]
   -------------------------- ------------- 2/3 [spotipy]
   ---------------------------------------- 3/3 [spotipy]

Note: you may need to restart the kernel to use updated packages.


## Authentication
Load Spotify API credentials from the .env file and authenticate using the Client Credentials flow.

In [16]:
import os
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from dotenv import load_dotenv

# Load credentials from .env
load_dotenv()

client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")

# Authenticate
auth_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(auth_manager=auth_manager)

print("Authentication successful!")

Authentication successful!


## Data Collection
Search for tracks across multiple genres using the Spotify API and collect audio features for each track.

In [18]:
# Define search queries across genres
search_queries = [
    ("pop", "pop hits"), ("pop", "top pop songs"),
    ("hip-hop", "hip hop"), ("hip-hop", "rap music"),
    ("rock", "rock music"), ("rock", "classic rock"),
    ("country", "country music"), ("country", "country hits"),
    ("latin", "latin music"), ("latin", "reggaeton"),
    ("r&b", "r&b music"), ("r&b", "soul music"),
    ("electronic", "electronic music"), ("electronic", "edm"),
    ("jazz", "jazz music"), ("jazz", "smooth jazz")
]

# Collect tracks
tracks = []
seen_ids = set()

for genre, query in search_queries:
    print(f"Collecting '{query}'...")
    results = sp.search(q=query, type="track", limit=10)
    
    for item in results["tracks"]["items"]:
        track_id = item.get("id", "")
        if track_id in seen_ids:
            continue
        seen_ids.add(track_id)

        # Numeric features
        popularity = item.get("popularity")
        duration_ms = item.get("duration_ms")
        track_number = item.get("track_number")
        disc_number = item.get("disc_number")

        # Categorical features
        explicit = item.get("explicit")
        album_type = item["album"].get("album_type", "")
        release_date = item["album"].get("release_date", "")
        release_year = release_date[:4] if release_date else None

        track = {
            "track_name": item.get("name", ""),
            "artist_name": item["artists"][0].get("name", "") if item["artists"] else "",
            "album_name": item["album"].get("name", ""),
            "popularity": popularity,
            "duration_ms": duration_ms,
            "track_number": track_number,
            "disc_number": disc_number,
            "release_year": release_year,
            "genre": genre,
            "explicit": explicit,
            "album_type": album_type,
            "track_id": track_id,
        }
        tracks.append(track)

df = pd.DataFrame(tracks)
print(f"\nTotal tracks collected: {len(df)}")
print(df.head())


Total tracks collected: 156
                 track_name     artist_name  \
0  Stateside + Zara Larsson  PinkPantheress   
1                    Golden         HUNTR/X   
2                  Pop Hits         Cashlin   
3                HOT TO GO!   Kidz Bop Kids   
4                  Pop Hits          xBOMBx   

                                          album_name popularity  duration_ms  \
0                                   Fancy Some More?       None       184761   
1  KPop Demon Hunters (Soundtrack from the Netfli...       None       194607   
2                                        High Lights       None        69308   
3                                        KIDZ BOP 50       None       185729   
4                                        10. 5. 2001       None       224010   

   track_number  disc_number release_year genre  explicit album_type  \
0            10            1         2025   pop     False      album   
1             4            1         2025   pop     False      

In [19]:
print(df.columns.tolist())
print(df.shape)
print(df[["popularity", "duration_ms", "explicit", "album_type", "release_year"]].head(10))

['track_name', 'artist_name', 'album_name', 'popularity', 'duration_ms', 'track_number', 'disc_number', 'release_year', 'genre', 'explicit', 'album_type', 'track_id']
(156, 12)
  popularity  duration_ms  explicit   album_type release_year
0       None       184761     False        album         2025
1       None       194607     False        album         2025
2       None        69308     False        album         2021
3       None       185729     False        album         2025
4       None       224010      True       single         2018
5       None       264906     False  compilation         1988
6       None       234466      True        album         2023
7       None       100930     False       single         2023
8       None       241733     False  compilation         2003
9       None       207653     False        album         2024


In [21]:
import random
import numpy as np

# Simulate realistic popularity scores (0-100)
# Based on recency and explicit content as rough proxies
random.seed(42)

for track in tracks:
    release_year = int(track["release_year"]) if track["release_year"] else 2000
    explicit = track["explicit"]
    
    # More recent tracks tend to be more popular in search results
    base = min(100, max(0, (release_year - 1990) * 2))
    # Add some randomness
    simulated = int(np.clip(random.gauss(base, 15), 0, 100))
    track["popularity"] = simulated

# Rebuild dataframe
df = pd.DataFrame(tracks)
print(f"Popularity null count: {df['popularity'].isna().sum()}")
print(df[["track_name", "release_year", "popularity"]].head(10))

Popularity null count: 0
                 track_name release_year  popularity
0  Stateside + Zara Larsson         2025          67
1                    Golden         2025          67
2                  Pop Hits         2021          60
3                HOT TO GO!         2025          80
4                  Pop Hits         2018          54
5                     Gypsy         1988           0
6            Hits Different         2023          70
7      Todays Top Hit Songs         2023          61
8  Joyride - Single Version         2003          22
9                   Popular         2024          69


## Note on Popularity Feature
The Spotify Web API's search endpoint and individual track endpoint no longer return 
popularity scores for new developer apps as of late 2024. As a workaround, popularity 
was simulated using a normal distribution centered around a base score derived from 
the track's release year, with added random variance (seed=42 for reproducibility). 
This approach reflects the general trend that more recently released tracks tend to 
appear more frequently in search results.

In [22]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

(156, 12)
track_name      object
artist_name     object
album_name      object
popularity       int64
duration_ms      int64
track_number     int64
disc_number      int64
release_year    object
genre           object
explicit          bool
album_type      object
track_id        object
dtype: object
track_name      0
artist_name     0
album_name      0
popularity      0
duration_ms     0
track_number    0
disc_number     0
release_year    0
genre           0
explicit        0
album_type      0
track_id        0
dtype: int64


## Export Data
Save the dataset to a CSV file for upload to cloud storage.

In [23]:
# Export to CSV
df.to_csv("data/spotify_tracks.csv", index=False)
print("Data saved to data/spotify_tracks.csv")
print(f"Final dataset: {df.shape[0]} rows x {df.shape[1]} columns")

Data saved to data/spotify_tracks.csv
Final dataset: 156 rows x 12 columns
